# Automated Curation Pipeline
This notebook demonstrates the end-to-end execution of the Bayesian Hierarchical Curation Framework. It transitions raw environmental data through ingestion, heuristic validation, and recursive MCMC modeling.

In [1]:
# =============================================================================
# 1. ENVIRONMENT SETUP & IMPORTS
# =============================================================================
import os
import sys
import yaml
import pandas as pd
import time
import warnings
import xarray as xr
from IPython.display import display

# Shift the notebook's working directory up to the project root.
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

from src.database.manager import DatabaseManager
from src.curation.disk_validator import validate_secchi_data
from src.curation.tube_validator import validate_tube_data
from src.curation.disk_model import BayesianDiskModel
from src.curation.tube_model import BayesianTubeModel

def load_config(config_path: str = "config/settings.yaml") -> dict:
    """Loads pipeline configuration parameters."""
    if not os.path.exists(config_path):
        raise FileNotFoundError(f"Configuration file not found at: {config_path}")
    with open(config_path, "r") as file:
        return yaml.safe_load(file)

print("Environment setup complete. Loading pipeline configuration...")
config = load_config()
print("\n✅ Configuration loaded successfully.")

Environment setup complete. Loading pipeline configuration...

✅ Configuration loaded successfully.


In [2]:
# =============================================================================
# 2. INGEST, SPLIT & VALIDATE DATA
# =============================================================================
print("Initializing full historical ingestion (1995 - Present)...")

# Extract the db_path directly from the config.
db_path = config.get("paths", {}).get("database", "data/master_registry.sqlite")

# Initialize the database manager and reset the existing schema
db_manager = DatabaseManager(db_path=db_path, overwrite=True)

# 1. Load and merge the raw GLOBE data with the site registry
df_merged = db_manager.load_and_merge_data(config=config)
print(f"\nTotal combined records loaded: {len(df_merged):,}")

# 2. Isolate the datastreams by instrument
df_disk_raw, df_tube_raw = db_manager.split_by_instrument(df_merged)
print(f" -> Isolated {len(df_disk_raw):,} potential Secchi Disk records.")
print(f" -> Isolated {len(df_tube_raw):,} potential Transparency Tube records.\n")

# 3. Apply physical boundaries for both instruments independently
print("Executing heuristic validations...")

# Pass the full configuration object and use .copy() to prevent Pandas warnings
df_disk_validated = validate_secchi_data(df_disk_raw.copy(), config)
df_tube_validated = validate_tube_data(df_tube_raw.copy(), config)

print("\n✅ Heuristic validation complete.")

Initializing full historical ingestion (1995 - Present)...
 -> Wiped existing database at data/master_registry.sqlite
 -> Synchronizing local data cache for years 1995-2025...
 -> Loading observations from 31 files...
 -> Generating deterministic sample tracking IDs (usid)...
 -> Normalizing site registry into relational tables...
 -> Executing temporal join across 166342 observations...

Total combined records loaded: 166,342
 -> Isolated 34,038 potential Secchi Disk records.
 -> Isolated 133,440 potential Transparency Tube records.

Executing heuristic validations...
  -> Flagged 15142 records bypassing Bayesian evaluation:
     - 9404 Right-Censored (Disk reached bottom visibly)
     - 5738 Heuristic Failures (Typos, negatives, depth contradictions)
  -> Flagged 71946 records bypassing Bayesian evaluation:
     - 67220 Right-Censored (Perfectly clear water)
     - 4726 Heuristic Failures (Typos, negatives, logical mismatches)

✅ Heuristic validation complete.


In [3]:
# =============================================================================
# 3. BAYESIAN PROBABILISTIC EVALUATION
# =============================================================================
# Suppress PyMC's standard warning about Potentials and Posterior Predictive Sampling — 
# temporal weights don't influence the simulated data checks.
warnings.filterwarnings("ignore", message="The effect of Potentials on other parameters is ignored")

print("Initializing Bayesian Secchi Disk Model...\n")
start_time_disk = time.time()

# Pass the full production config directly
disk_model = BayesianDiskModel(config)
df_disk_curated = disk_model.evaluate(df_disk_validated, window_years=10)

disk_elapsed = (time.time() - start_time_disk) / 60
print(f" -> Secchi Disk MCMC sampling complete in {disk_elapsed:.2f} minutes.\n", flush=True)

print("Initializing Bayesian Transparency Tube Model...\n")
start_time_tube = time.time()

# Pass the full production config directly
tube_model = BayesianTubeModel(config)
df_tube_curated = tube_model.evaluate(df_tube_validated, window_years=10)

tube_elapsed = (time.time() - start_time_tube) / 60
print(f" -> Transparency Tube MCMC sampling complete in {tube_elapsed:.2f} minutes.\n", flush=True)

total_elapsed = (time.time() - start_time_disk) / 60
print(f"✅ Bayesian evaluation complete. Total execution time: {total_elapsed:.2f} minutes.\n")


# Export traces
print("Exporting traces to NetCDF...")

trace_dir = './output/traces'
os.makedirs(trace_dir, exist_ok=True)

disk_path = os.path.join(trace_dir, 'disk_trace.nc')
tube_path = os.path.join(trace_dir, 'tube_trace.nc')

def force_save_trace(trace, filepath):
    """
    Forces a save by clearing xarray's internal file locks and
    deleting the existing file before writing the new one.
    """
    # 1. Clear xarray's backend file cache to release any lingering NetCDF file locks
    xr.backends.file_manager.FILE_CACHE.clear()

    # 2. Safely delete the file if it already exists
    if os.path.exists(filepath):
        try:
            os.remove(filepath)
            print(f"  -> Cleared previous file: {filepath}")
        except Exception as e:
            print(f"  -> Warning: Could not delete {filepath}. Error: {e}")

    # 3. Save the new trace cleanly
    trace.to_netcdf(filepath)
    print(f"✅ Trace saved successfully to {filepath}")

force_save_trace(disk_model.trace, disk_path)
force_save_trace(tube_model.trace, tube_path)

Initializing NUTS using jitter+adapt_diag...


Initializing Bayesian Secchi Disk Model...



Multiprocess sampling (4 chains in 4 jobs)
NUTS: [global_mu, global_sigma, water_source_offset, water_source_sigma, site_offset, obs_sigma]


Output()

Sampling 4 chains for 2_000 tune and 4_000 draw iterations (8_000 + 16_000 draws total) took 62 seconds.


 -> Secchi Disk MCMC sampling complete in 1.08 minutes.

Initializing Bayesian Transparency Tube Model...



Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [global_mu, global_sigma, water_source_offset, water_source_sigma, site_offset, obs_sigma]


Output()

Sampling 4 chains for 2_000 tune and 4_000 draw iterations (8_000 + 16_000 draws total) took 2164 seconds.
There were 8 divergences after tuning. Increase `target_accept` or reparameterize.
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details


 -> Transparency Tube MCMC sampling complete in 36.27 minutes.

✅ Bayesian evaluation complete. Total execution time: 37.35 minutes.

Exporting traces to NetCDF...
  -> Cleared previous file: ./output/traces/disk_trace.nc
✅ Trace saved successfully to ./output/traces/disk_trace.nc
  -> Cleared previous file: ./output/traces/tube_trace.nc
✅ Trace saved successfully to ./output/traces/tube_trace.nc


In [4]:
# =============================================================================
# 4. EXPORT CURATED DATA & AUDIT LOGS
# =============================================================================

# Reuse db_manager from Cell 2 to preserve the column mappings required for table normalization

print("Exporting Secchi Disk artifacts...")
# Export curated data to the SQLite Master Registry
db_manager.export_curated_data(df_disk_curated, table_name='measurements_disk')

# Extract the flagged directory
flagged_dir = config.get("paths", {}).get("flagged_output", "data/flagged/")
db_manager.export_audit_log(df_disk_curated, output_path=os.path.join(flagged_dir, 'flagged_disk.csv'))

print("\nExporting Transparency Tube artifacts...")
db_manager.export_curated_data(df_tube_curated, table_name='measurements_tube')
db_manager.export_audit_log(df_tube_curated, output_path=os.path.join(flagged_dir, 'flagged_tube.csv'))

print("\n✅ All data successfully routed and exported to the Master Registry and Audit Logs.")

Exporting Secchi Disk artifacts...

Exporting Transparency Tube artifacts...

✅ All data successfully routed and exported to the Master Registry and Audit Logs.


In [5]:
# =============================================================================
# 5. DATA CURATION DIAGNOSTICS & EXECUTION SUMMARY
# =============================================================================
def analyze_dataframe(df, instrument_name):
    if df is None or df.empty:
        print(f"⚠️ No data available for {instrument_name}.\n")
        return

    # Dynamically determine which flags exist in this specific dataframe
    group_cols = ['passed_heuristics', 'is_statistical_outlier']
    if 'is_censored' in df.columns:
        group_cols.append('is_censored')

    # Group by the available boolean flags to get raw counts
    summary_df = df.groupby(group_cols).size().reset_index(name='record_count')

    # Check for right-censoring first, as these valid limits are intentionally 
    # flagged to bypass the continuous Bayesian model
    def categorize(row):
        is_censored = bool(row.get('is_censored', False))
        passed_heur = bool(row['passed_heuristics'])
        is_outlier = bool(row.get('is_statistical_outlier', False))

        if is_censored:
            return "Valid (Right-Censored / Hit Bottom)"
        elif not passed_heur:
            return "Failed Heuristics (Physical/Logic Error)"
        elif is_outlier:
            return "Failed Bayesian (Statistical Outlier)"
        else:
            return "Pristine (Passed All)"

    total_records = summary_df['record_count'].sum()
    summary_df['percentage'] = (summary_df['record_count'] / total_records * 100).round(2)
    summary_df['category'] = summary_df.apply(categorize, axis=1)

    # Aggregate by category in case multiple boolean combos map to the same string
    final_summary = summary_df.groupby('category').agg({
        'record_count': 'sum',
        'percentage': 'sum'
    }).reset_index()

    # Sort for clean presentation (Pristine -> Censored -> Bayesian Fails -> Heuristic Fails)
    sort_order = {
        "Pristine (Passed All)": 1,
        "Valid (Right-Censored / Hit Bottom)": 2,
        "Failed Bayesian (Statistical Outlier)": 3,
        "Failed Heuristics (Physical/Logic Error)": 4
    }

    # Map sort order; unknown categories default to 5 (appended last)
    final_summary['sort_key'] = final_summary['category'].map(sort_order).fillna(5)
    final_summary = final_summary.sort_values('sort_key').drop(columns=['sort_key']).reset_index(drop=True)

    # Render the final tables
    print(f"=== {instrument_name} ===")
    print(f"Total Records Evaluated: {total_records:,}")
    display(final_summary[['category', 'record_count', 'percentage']])
    print("\n")

# Generate summaries directly from the in-memory DataFrames
analyze_dataframe(df_disk_curated, 'Secchi Disk Diagnostics')
analyze_dataframe(df_tube_curated, 'Transparency Tube Diagnostics')

=== Secchi Disk Diagnostics ===
Total Records Evaluated: 34,038


,category,record_count,percentage
0,Pristine (Passed All),18737,55.05
1,Valid (Right-Censored / Hit Bottom),9404,27.63
2,Failed Bayesian (Statistical Outlier),159,0.47
3,Failed Heuristics (Physical/Logic Error),5738,16.86




=== Transparency Tube Diagnostics ===
Total Records Evaluated: 133,440


,category,record_count,percentage
0,Pristine (Passed All),58959,44.18
1,Valid (Right-Censored / Hit Bottom),67220,50.37
2,Failed Bayesian (Statistical Outlier),2535,1.90
3,Failed Heuristics (Physical/Logic Error),4726,3.54


---

# End of Curation Pipeline

If the cells above executed without errors, the Bayesian Hierarchical Curation Framework has completed its run.

### Expected Outputs:
* **SQLite Registry:** Valid measurements have been successfully merged into the permanent `data/master_registry.sqlite` database.
* **Audit Logs:** Flagged anomalies and statistical outliers have been routed to the `data/flagged/` directory for human review.

### Next Steps
With the core curation cycle complete, proceed to the analytical notebooks:

* **`02_model_diagnostics.ipynb`**: Review the PyMC traces, Gelman-Rubin statistics, Posterior Predictive Checks (PPCs), and quantitative information criteria (WAIC/LOO) to ensure the MCMC chains converged properly and assess predictive model fit.
* **`03_exploratory_analysis.ipynb`**: Analyze the curated and harmonized dataset to quantify spatio-temporal trends in global water transparency.

---